In [31]:
import requests
import json
import re
import pandas as pd
import numpy as np

def get_uniprot(accession):
  '''
  define http_function to get the data from Uniprot API
  '''
  http_function = requests.get
  endpoint = f'https://rest.uniprot.org/uniprotkb/{accession}.json'
  http_args = {'headers': {'Accept': 'application/json'}}
  return http_function(endpoint, **http_args)

print(endpoint.

def uniprot_parse_response(resp: dict):
    '''
    parse response from Uniprot and output
    organism, geneInfo, sequenceInfo, type

    do not forget to include error handling (e.g. for key errors)
    '''
    try:
        data = resp.json()

        if resp.status_code != 200:
            acc = resp.url.split('/')[-1].replace('.json', '')
            if 'messages' in data:
                output = {acc: 'error:' + '; '.join(data['messages'])}
            else:
                output = {acc: 'error:uniprot request failed'}
            return output

        acc = data['primaryAccession']
        output = {
            acc: {
                'organism': data['organism']['scientificName'],
                'geneInfo': data['genes'],
                'sequenceInfo': data['sequence'],
                'type': 'protein'
            }
        }

    except Exception:
        try:
            acc = resp.url.split('/')[-1].replace('.json', '')
            output = {acc: 'error:failed to parse uniprot response'}
        except Exception:
            output = {'unknown': 'error:failed to parse uniprot response'}

    return output


def get_ensembl(id):
  '''
  define http_function to get the data from Ensembl API
  '''
  http_function = requests.get
  endpoint = f'https://rest.ensembl.org/lookup/id/{id}'
  http_args = {'headers': {'Accept': 'application/json'}}
  return http_function(endpoint, **http_args)


def ensembl_parse_response(resp: dict):
  '''
  parse Ensembl response and output
  object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

  do not forget to include error handling (e.g. for key errors)
  '''
  try:
      data = resp.json()

      if resp.status_code != 200:
          ens_id = resp.url.split('/')[-1]
          if 'error' in data:
              output = {ens_id: 'error:' + data['error']}
          else:
              output = {ens_id: 'error:ensembl request failed'}
          return output

      ens_id = data['id']
      output = {
          ens_id: {
              'object_type': data['object_type'],
              'species': data['species'],
              'assembly_name': data['assembly_name'],
              'biotype': data['biotype'],
              'display_name': data['display_name'],
              'id': data['id'],
              'db_type': data['db_type'],
              'description': data['description'],
              'source': data['source'],
              'canonical_transcript': data['canonical_transcript']
          }
      }

  except Exception:
      try:
          ens_id = resp.url.split('/')[-1]
          output = {ens_id: 'error:failed to parse ensembl response'}
      except Exception:
          output = {'unknown': 'error:failed to parse ensembl response'}

  return output


def main(ids: list):
  '''
  Function that iterates over all the provided IDs and parses them into dict,
  transforms into pandas.DataFrame, and return it
  {ID : info from parse_response(), ...}

  If ID is incorrect, it should return {ID : error message}
  '''
  output = {}

  uniprot_pattern = r'^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9][A-Z0-9]{3}[0-9])$'
  ensembl_pattern = r'^ENS[A-Z0-9]*\d+$'

  for i in ids:
      if re.match(uniprot_pattern, i):
          output.update(uniprot_parse_response(get_uniprot(i)))
      elif re.match(ensembl_pattern, i):
          output.update(ensembl_parse_response(get_ensembl(i)))
      else:
          output[i] = 'error:unknown database'

  return output




In [32]:
get_uniprot('P11473')

<Response [200]>

In [33]:
get_uniprot('helloworld')

<Response [400]>

In [34]:
get_uniprot('helloworld').json()


{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [35]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

In [36]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [37]:
get_ensembl('helloworld')

<Response [400]>

In [38]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [39]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'species': 'mus_musculus',
  'assembly_name': 'GRCm39',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'db_type': 'core',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'source': 'ensembl_havana',
  'canonical_transcript': 'ENSMUST00000044620.11'}}

In [40]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'},
 'Q91XI3': {'organism': 'Ictidomys tridecemlineatus',
  'geneInfo': [{'geneName': {'value': 'INS'}}],
  'sequenceInfo': {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVE